In [1]:
from pathlib import Path
import numpy as np
from astropy import units as u
from astropy.table import Table

from telescope import *
from spectrograph import *
from detector import *
import synphot
from etc.etc import SpecETC

from matplotlib import pyplot as plt

LRIS-B-600: Reading spectrograph efficiency from /Users/joshw/git/SpecETC/data/LRIS-Blue/600_mirr_eff.dat
IMX183: Reading detector efficiency from /Users/joshw/git/SpecETC/data/SonyIMX/QE.csv
IMX533: Reading detector efficiency from /Users/joshw/git/SpecETC/data/SonyIMX/QE.csv


In [2]:
pickles_readme_file = Path('~/pysynphot_models/grid/pickles/AA_README').expanduser()
with open(pickles_readme_file) as f:
    readme = f.readlines()
pickles_readme_table = Table.read(readme[113:193], format='ascii', names=('file', 'SPTYPE', 'Teff'))

In [3]:
types = [str(spt) for spt in set(pickles_readme_table['SPTYPE'])]
for ty in ['O', 'B', 'A', 'F', 'G', 'K', 'M']:
    subtypes = [st for st in types if st[0] == ty]
    print(ty, subtypes)

O ['O9V', 'O5V', 'O8III']
B ['B2IV', 'B2II', 'B8V', 'B5III', 'B1V', 'B5I', 'B5II', 'B1-2III', 'B3V', 'B0I', 'B9III', 'B0V', 'B5-7V', 'B8I', 'B6IV']
A ['A0IV', 'A0V', 'A2V', 'A0I', 'A5III', 'A5V', 'A3V', 'A0III', 'A4-7IV']
F ['F2II', 'F5III', 'F8I', 'F0III', 'F5IV', 'F5V', 'F0I', 'F0-2IV', 'F2V', 'F0II', 'F8V', 'F0V', 'F5I', 'F8IV']
G ['G2IV', 'G2V', 'G0V', 'G5II', 'G8III', 'G8V', 'G8IV', 'G0I', 'G5I', 'G0IV', 'G5V', 'G8I', 'G0III', 'G5IV', 'G5III']
K ['K2V', 'K0-1II', 'K4I', 'K0III', 'K3IV', 'K0V', 'K2I', 'K0IV', 'K7V', 'K5III', 'K3-4II', 'K3III', 'K5V', 'K1IV']
M ['M5V', 'M0V', 'M0III', 'M10III', 'M3II', 'M2V', 'M5III', 'M2I', 'M4V']


In [4]:
sptype = 'F8V'
files = pickles_readme_table[pickles_readme_table['SPTYPE'] == sptype]
file = str(files[0]['file'])+'.fits'
p = Path(f'~/pysynphot_models/grid/pickles/').expanduser()
if file.index('uk') > 0:
    file = p / 'dat_uvk' / file
else:
    file = p / 'dat_uvi' / file
print(f"{file}: {file.exists()}")

/Users/joshw/pysynphot_models/grid/pickles/dat_uvk/pickles_uk_20.fits: True


In [5]:
mag = 12
vega = synphot.SourceSpectrum.from_vega()*10**(-mag/2.5)
star = synphot.SourceSpectrum.from_file(str(file))*10**(-mag/2.5)

In [6]:
spec, var, im = SpecETC(star, exptime=300*u.second, seeing=3.0*u.arcsec,
                        telescope=Newtonian12, spectrograph=Alpy600, detector=IMX183)

UnitConversionError: Can only apply 'add' function to dimensionless quantities when other argument is not a quantity (unless the latter is all zero/infinity/nan).

In [ ]:
# sky_file = Path(f'~/pysynphot_models/nonstellar/skybg_50_10.dat').expanduser()
# with open(sky_file, 'r') as f:
#     sky_contents = f.readlines()
# sky_table = Table.read(sky_contents[13:], format='ascii')
# # synphot.units.PHOTLAM is phot/s/cm^2/A
# # table is phot/s/nm/arcsec^2/m^2
# A = sky_table['nm']*10
# sky_table.add_column(A, name='A')
# sky_table.remove_column('nm')
# # synphot.units.PHOTLAM is phot/s/cm^2/A
# # table is phot/s/nm/arcsec^2/m^2
# photlam = sky_table['phot/s/nm/arcsec^2/m^2']/1e5 # phot/s/A/arcsec^2/cm^2
# sky_table.add_column(photlam, name='photlam/acrsec^2')
# sky_table.remove_column('phot/s/nm/arcsec^2/m^2')
# sky_file2 = Path(f'~/pysynphot_models/skybg_50_10_photlam.dat').expanduser()
# sky_table.write(sky_file2, overwrite=True, format='ascii')

# sky = synphot.SourceSpectrum.from_file(str(sky_file2),
#                                        flux_unit=synphot.units.PHOTLAM)
# sky.plot()

In [ ]:
telescopes = [Newtonian12, ACF14reduced, SVX152]
nexp = [9, 6, 3]
exptimes = [300, 480, 900]
spectra = {}
variance = {}
for t,tel in enumerate(telescopes):
    spec, var, im = SpecETC(star, exptime=exptimes[t]*u.second, seeing=3.0*u.arcsec,
                            telescope=tel, spectrograph=Alpy600, detector=IMX183, plot=False)
    spectra[tel.name] = spec*nexp[t]
    variance[tel.name] = var*nexp[t]**0.5

In [ ]:
Alpy600.generate_binset(IMX183)
plt.figure(figsize=(12,4))
plt.title(f'{sptype} star at {mag:.1f} mag')
peakSNR = 0
for t,tel in enumerate(telescopes):
    SNR = spectra[tel.name]/variance[tel.name]**0.5
    label = f"{tel.name} {nexp[t]:d}x{exptimes[t]:.0f}s={nexp[t]*exptimes[t]:.0f}s"
    plt.plot(Alpy600.binset, SNR, alpha=0.5, label=label)
    peakSNR = peakSNR if max(SNR) <= peakSNR else max(SNR)
plt.ylabel('SNR')
plt.xlabel('Wavelength (A)')
plt.xlim(min(Alpy600.binset), max(Alpy600.binset))
plt.ylim(0,1.1*peakSNR)
plt.grid()
plt.legend(loc='best')
plt.show()

In [ ]:
spec, var, im = SpecETC(star, exptime=300*u.second, seeing=3.0*u.arcsec,
                        telescope=Newtonian12, spectrograph=Alpy600, detector=IMX183)

In [ ]:
spec, var, im = SpecETC(star, exptime=600*u.second, seeing=3.0*u.arcsec,
                        telescope=ACF14reduced, spectrograph=Alpy600, detector=IMX183)

In [ ]:
spec, var, im = SpecETC(star, exptime=1000*u.second, seeing=3.0*u.arcsec,
                        telescope=SVX152, spectrograph=Alpy600, detector=IMX183)